# Linear-algebra kernels

The numeric core of every penalized learner is a small dense linear solve. Two
ingredients make it fast and stable: the [`faer`](https://faer-rs.github.io/)
dense linear-algebra library and a cached `GramSolver` that selects among three
factorizations according to the conditioning of the matrix it is given.

## The `GramSolver`

A penalized Gram matrix $G = X^\top W X + \lambda K$ is symmetric, and positive
definite when the penalty or the design has full rank. The `GramSolver`
factorizes it once and exposes `solve_into_slice(rhs, out)` for repeated
right-hand sides:

- The factorization is computed at `prepare_*` time and chosen from three
  routes, described under [](#choosing-a-factorization) below.
- Each boosting step provides a fresh `rhs` $= X^\top W u$ and back-solves in
  $O(p^2)$.

Because the boosting loop changes only `u`, this amortizes the $O(p^3)$
factorization over all $m_{\text{stop}}$ iterations.

## The `faer` library

`faer` provides cache-friendly, SIMD-accelerated dense kernels (matmul, Cholesky,
QR) in pure Rust with no external BLAS/LAPACK dependency — so wheels are
self-contained and portable. The performance-critical operations are kept in
`faer`'s column-major `Mat<f64>` representation:

- $X^\top W X$ is formed as a single weighted matmul.
- $W X^\top$ is precomputed and stored, so the per-step right-hand side
  $X^\top W u$ is one matrix–vector product.

Conversions between `ndarray` (used for ergonomic construction) and `faer` (used
for the hot kernels) happen once at preparation time, never in the loop.

## The `df` to `λ` solve

Calibrating a penalty to a target `df` requires solving
$\mathrm{df}(\lambda) = \mathrm{df}^\star$ where
$\mathrm{df}(\lambda) = \operatorname{tr}(2H - HH^\top)$. The implementation
diagonalizes the penalized problem once — a generalized eigendecomposition of
$(K, X^\top W X)$ giving eigenvalues $d_\ell$ — after which

$$
\mathrm{df}(\lambda) = \sum_\ell \frac{2}{1+\lambda d_\ell} - \frac{1}{(1+\lambda d_\ell)^2}
$$

is an $O(p)$ evaluation. Brent's method — the bracketed `zeroin` root-finder,
shared with `stabsel.rs` — then locates $\lambda$ on $[0, 10^{15}]$ to a
tolerance of $\sqrt{\varepsilon}$, which is how `mboost:::df2lambda` calls
`uniroot`. The same routine, `df_to_lambda(X, K, w, df)`, is used by
every penalized learner and is re-invoked per cross-validation fold with the
fold's weights.

## The `GramSolver` API

`GramSolver` is an enum over the three factorizations, defined in `linalg.rs`.
The variant that was chosen is therefore visible to callers, and the execution
report counts each one:

```rust
pub enum GramSolver {
    Cholesky(CachedCholesky),
    Lu(CachedLu),
    RegularizedCholesky { factor: CachedCholesky, jitter: f64 },
}
```

The operations used by the boosting loop are:

| Method | Purpose |
|---|---|
| `try_new(gram)` | Factorize, choosing a route; returns a typed error rather than a solver that emits `NaN`. |
| `solve_into_slice(b, x_out)` | Solve $G x = b$ into a caller-owned buffer. Zero-allocation hot path. |
| `solve(b)` / `solve_mat(b)` | Allocating variants for a vector and a matrix right-hand side. |
| `is_cholesky()` | Whether the plain Cholesky route was taken. |
| `regularization_jitter()` | `Some(ε)` when the matrix was regularized, `None` otherwise. |
| `cached_factor_bytes()` | Size of the retained factor, for the memory report. |

Both Cholesky routes solve by forward substitution ($L z = \text{rhs}$) then
back substitution ($L^\top x = z$), each $O(p^2/2)$ flops and neither requiring
pivoting. The LU route pivots by row, which is why it stores a permutation
alongside its factors and reports twice the cached bytes.

(choosing-a-factorization)=
## Choosing a factorization

$G$ is not always positive definite. With $\lambda = 0$ and a rank-deficient
$X$, or with a small $\lambda$ and a $K$ that has a null space, it is only
semi-definite; `bmrf` contributes a $K$ with negative eigenvalues, which can make
it indefinite. `GramSolver::try_new` therefore tries the routes in order of cost:

The routing below condenses `GramSolver::try_new_unmeasured` in
`crates/mboostrs-core/src/linalg.rs`; the validity checks on shape and finiteness
are omitted for brevity.

```rust
// 1. Cholesky, but only if the factor is not numerically rank-deficient.
match CachedCholesky::new(gram) {
    Ok(chol) if !chol.is_numerically_rank_deficient() => {
        return Ok(GramSolver::Cholesky(chol));
    }
    Ok(_) => rank_deficient = true,   // skip LU, go straight to jitter
    Err(_) => {}                      // not positive definite; try LU
}

// 2. LU with partial pivoting, for an indefinite but invertible G.
if !rank_deficient {
    let lu = CachedLu::new(gram)?;
    if lu.has_acceptable_inverse_residual() {
        return Ok(GramSolver::Lu(lu));
    }
}

// 3. Diagonal jitter, scaled to the magnitude of G.
let scale = gram.iter().fold(0.0, |s, v| s.max(v.abs())).max(1.0);
let mut jitter = 1e-12 * scale;
for _ in 0..14 {
    let regularized = gram + jitter * I;
    if let Ok(factor) = CachedCholesky::new(&regularized) {
        if !factor.is_numerically_rank_deficient() {
            return Ok(GramSolver::RegularizedCholesky { factor, jitter });
        }
    }
    jitter *= 10.0;
}
Err(MboostrsError::Solver("matrix is singular and diagonal regularization failed"))
```

**Pivot-size check.** For $G = L L^\top$ the pivots are
$L_{jj}^2$, so a matrix can factorize while its smallest pivot sits at the
double-precision resolution of its largest. `diag(1, 10^{-20})` is positive
definite, factorizes without error, and then solves to $x_2 = 10^{20}$.
`CachedCholesky::is_numerically_rank_deficient` rejects that case by comparing
the smallest and largest entries of the $L$ diagonal against
$\sqrt{\varepsilon_{\text{mach}}} \approx 1.5\times10^{-8}$, which places the
threshold at a matrix condition number of about
$1/\varepsilon_{\text{mach}} \approx 4.5\times10^{15}$ — the point at which no
digit of the solve is reliable. Systems that are ill-conditioned but still usable
($\kappa \approx 10^{10}$) stay on the fast path.

**LU bypass for a rank-deficient $G$.** `CachedLu`'s
acceptance test scales its residual tolerance by the magnitude of the computed
inverse, so it admits a solve that is accurate in a relative sense while being
enormous in an absolute one. That is the correct test for an indefinite matrix,
and the wrong one for a near-singular matrix, so the near-singular case bypasses
it. An indefinite $G$ fails Cholesky outright, never reaches the guard, and still
reaches LU.

**Scale-dependent jitter.** It starts at
$\varepsilon = 10^{-12}\,\lVert G \rVert_{\max}$ and multiplies by 10 for at most
14 attempts, so it spans thirteen decades of the matrix's own scale. Each
candidate factor must also clear the rank-deficiency check: enough jitter to
factorize is not enough jitter to be solvable. The jittered solution approximates
the Moore–Penrose pseudoinverse solution, and matches R's behavior when `mboost`
encounters a singular Gram matrix.

Which route a given matrix takes is recorded in the execution report as
`cholesky_solver_count`, `lu_solver_count`, and `regularized_solver_count`, with
`max_solver_jitter` giving the largest $\varepsilon$ applied.

### Worked example: jitter for a rank-deficient design

Consider a `bols(x1, x2, x3)` learner where the third column is an exact linear
combination of the first two ($x_3 = x_1 + x_2$) — a design mistake that happens
in practice, when a derived feature is re-included alongside its inputs. With
$n=4$ observations and equal weights $w_i=1$:

| $i$ | $x_1$ | $x_2$ | $x_3 = x_1+x_2$ |
|-----|-------|-------|-----------------|
| 1   | 1.0   | 1.0   | 2.0             |
| 2   | 2.0   | −1.0  | 1.0             |
| 3   | 0.0   | 3.0   | 3.0             |
| 4   | −1.0  | 1.0   | 0.0             |

$G = X^\top X$ is $3\times3$ with **rank 2**: its null space is spanned by
$(1, 1, -1)$, since $x_1 + x_2 - x_3 = 0$ for every row regardless of weights.
Cholesky fails on this $G$, and the jittered solve agrees with the
Moore–Penrose pseudoinverse to the tolerance printed below — close enough to be a
safe substitute for failing the fit outright.

The cell below reproduces both the jitter progression and the near-singular case
the rank-deficiency guard exists to catch.

In [1]:
import numpy as np
from scipy.linalg import cho_factor, cho_solve

X = np.array([
    [1.0, 1.0, 2.0],
    [2.0, -1.0, 1.0],
    [0.0, 3.0, 3.0],
    [-1.0, 1.0, 0.0],
])
w = np.ones(4)
G = X.T @ (w[:, None] * X)
print(f"rank(G) = {np.linalg.matrix_rank(G)} of {G.shape[0]}, det(G) = {np.linalg.det(G):.3e}")

try:
    cho_factor(G)
    raise AssertionError("Cholesky should have failed on a singular G")
except np.linalg.LinAlgError:
    print("Cholesky failed on the un-jittered (singular) G, as expected")

u = np.array([0.5, -0.2, 0.3, -0.1])   # an arbitrary pseudo-residual
rhs = X.T @ (w * u)                     # a real, consistent right-hand side

# Mirror the engine: jitter is scaled to |G|_max and multiplied by 10, for at
# most 14 attempts.
scale = max(np.abs(G).max(), 1.0)
eps = 1e-12 * scale
for _ in range(14):
    try:
        chol = cho_factor(G + eps * np.eye(3))
        break
    except np.linalg.LinAlgError:
        eps *= 10.0
beta_jitter = cho_solve(chol, rhs)
beta_pinv = np.linalg.pinv(G) @ rhs
max_diff = np.abs(beta_jitter - beta_pinv).max()
print(f"|G|_max = {scale:.1f}, jitter eps used: {eps:.0e}")
print(f"jittered beta: {beta_jitter}")
print(f"pinv beta:     {beta_pinv}")
print(f"max abs diff:  {max_diff:.2e}")
assert max_diff < 1e-3

# The near-singular case: Cholesky succeeds, and that is the problem.
near_singular = np.diag([1.0, 1e-20])
cho_factor(near_singular)                      # no error
L = np.linalg.cholesky(near_singular)
ratio = np.diag(L).min() / np.diag(L).max()
unguarded = cho_solve(cho_factor(near_singular), np.ones(2))
print(f"\nnear-singular diag(1, 1e-20): Cholesky succeeds")
print(f"  min/max L diagonal = {ratio:.1e}  (guard threshold {np.sqrt(np.finfo(float).eps):.1e})")
print(f"  unguarded solve    = {unguarded}")
print(f"  rejected by guard  = {ratio <= np.sqrt(np.finfo(float).eps)}")

rank(G) = 2 of 3, det(G) = 0.000e+00
Cholesky failed on the un-jittered (singular) G, as expected
|G|_max = 14.0, jitter eps used: 1e-11
jittered beta: [0.00686165 0.06568518 0.07255012]
pinv beta:     [0.00686275 0.06568627 0.07254902]
max abs diff:  1.10e-06

near-singular diag(1, 1e-20): Cholesky succeeds
  min/max L diagonal = 1.0e-10  (guard threshold 1.5e-08)
  unguarded solve    = [1.e+00 1.e+20]
  rejected by guard  = True


For the rank-deficient $G$, the jittered Cholesky solve and the Moore–Penrose
pseudoinverse solve agree to 1.10e-06 above, so the jitter is a cheap substitute
for detecting and special-casing a rank-deficient design.

The second block is the case that motivated the rank-deficiency guard.
`diag(1, 10^{-20})` factorizes without error and solves to $10^{20}$; the ratio
of its smallest to largest $L$ diagonal entry is $10^{-10}$, below the
$1.5\times10^{-8}$ threshold, so `try_new` regularizes it instead of returning
that answer. Before the guard existed this matrix took the plain Cholesky route.
Whether a rank-deficient $G$ fails Cholesky or merely factorizes with a tiny
pivot depends on rounding, and therefore on the target: the same duplicate-column
design was exactly singular on aarch64 and singular only to within one unit in
the last place on an FMA-enabled local benchmark build.

## The generalized eigendecomposition for `df_to_lambda`

The `df_to_lambda` routine requires the generalized eigenvalues $d_\ell$ of
$(K, X^\top W X)$. The implementation avoids a generalized eigensolver by
transforming to a standard symmetric problem:

1. Compute the Cholesky factor of $X^\top W X$: $X^\top W X = L_X L_X^\top$.
2. Form the **transformed penalty** $\tilde K = L_X^{-1} K L_X^{-\top}$.
3. Compute the eigenvalues $d_\ell$ of $\tilde K$ via `faer`'s symmetric
   eigendecomposition (tridiagonal QR or divide-and-conquer).

Step 2 gives $\tilde K v = d v$ if and only if $K u = d (X^\top W X) u$ with
$u = L_X^{-\top} v$. This is the standard transformation for symmetric
generalized eigenvalue problems with a positive-definite right-hand matrix.

Once the eigenvalues $d_\ell$ are computed, each evaluation of
$\mathrm{df}(\lambda)$ is an $O(p)$ sum. The eigendecomposition itself is
$O(p^3)$ but performed only once per learner (or per CV fold when weights
change).

## Numerical conventions matched to R

A few low-level choices exist solely to reproduce R bit-for-(nearly-)bit:

- **`optimize`-style 1-D minimization.** Family offsets and nuisance parameters
  that R obtains via `optimize` are computed with a Brent minimizer using the same
  bracket and tolerance (`.Machine$double.eps^0.25`), so the offsets agree to
  $\sim 10^{-12}$.
- **`digamma`/`lgamma`.** Count and Gamma families need the digamma function; it
  is implemented via the recurrence-plus-asymptotic expansion that matches R's to
  $\sim 10^{-12}$.
- **Softmax stabilization.** The multinomial denominator subtracts the per-row max
  (including the reference class's implicit 0) and adds the reference term as
  $e^{-m_i}$; see [](kronecker_multinomial.ipynb).

These are required: without them the parity tests (which assert
agreement to $\sim 10^{-6}$ or tighter against R) fail.

## See also

- [](../methodology/penalties_df) — the df-to-lambda calibration that uses
  this generalized eigendecomposition.
- [](engine.ipynb) — how the GramSolver's cached factorization feeds the engine's
  per-step back-solve.
- [](kronecker_multinomial.ipynb) — the Kronecker-product Gram matrix for
  Multinomial models.
- [](parity_and_bugs.ipynb) — why numerical conventions (digamma, Brent, jitter)
  matter for R parity.
- [](../methodology/functional_gradient_boosting) — how GramSolver's
  back-solve feeds into each boosting step's learner fit.